In [ ]:
import numpy as np
import cv2
import os
import random
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense,Activation,Dropout,BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau


In [ ]:
def find_Class(directory_path):
    if not os.path.exists(directory_path):
        raise ValueError(f"The directory '{directory_path}' does not exist.")
    if not os.path.isdir(directory_path):
        raise ValueError(f"The path '{directory_path}' is not a directory.")
    
    # Get a list of all entries in the directory
    all_entries = os.listdir(directory_path)
    
    # Filter out only directories
    folders = [entry for entry in all_entries if os.path.isdir(os.path.join(directory_path, entry))]
    
    return folders


In [ ]:
DIRECTORY= r'/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/train'
CATAGORIES= []
try:
    folders = find_Class(DIRECTORY)
    print(f"Directories in '{DIRECTORY}':")
    for folder in folders:
        CATAGORIES.append(folder)
except ValueError as e:
    print(e)

CATAGORIES


In [ ]:
data=[]

for categories in CATAGORIES:
    folder=os.path.join(DIRECTORY,categories)
    label=CATAGORIES.index(categories)
    
    
    for img in os.listdir(folder):
        img=os.path.join(folder,img)
        img_arr=cv2.imread(img)
        if img_arr is not None:  # Check if the image is successfully loaded
            img_arr = cv2.resize(img_arr, (100, 100))
            data.append([img_arr, label])
        else:
            print(f"Failed to load image {img}")


In [ ]:
len(data)

In [ ]:
DIRECTORY= r'/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/test'
for categories in CATAGORIES:
    folder=os.path.join(DIRECTORY,categories)
    label=CATAGORIES.index(categories)
    
    
    for img in os.listdir(folder):
        img=os.path.join(folder,img)
        img_arr=cv2.imread(img)
        if img_arr is not None:  # Check if the image is successfully loaded
            img_arr = cv2.resize(img_arr, (100, 100))
            data.append([img_arr, label])
        else:
            print(f"Failed to load image {img}")

In [ ]:
len(data)

In [ ]:
random.shuffle(data)

In [ ]:
x=[]
y=[]


for features,label in data:
    x.append(features)
    y.append(label)

In [ ]:
X= np.array(x)
Y=np.array(y)

In [ ]:
X=X/255

In [ ]:
X.shape

In [ ]:
Y.shape

In [ ]:
model=Sequential()
model.add( Conv2D(32,(3,3),input_shape=X.shape[1:],activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.5))

model.add( Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.5))

model.add( Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(BatchNormalization())
model.add(Dropout(0.5))


model.add(Flatten())
model.add(Dense(7,activation='softmax'))

model.summary()

In [ ]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
checkpoint=ModelCheckpoint(r'fer.keras',
                          monitor='val_accuracy',
                          mode='max',
                          save_best_only=True,
                          verbose=1)
earlystop=EarlyStopping(monitor='val_accuracy',
                        mode='max',
                       min_delta=0.001,
                       patience=10,
                       verbose=1,
                       restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3
)

callbacks=[checkpoint,earlystop]


In [ ]:
history = model.fit(X, Y, epochs=100, validation_split=0.25, callbacks=callbacks)

In [ ]:
import matplotlib.pyplot as plt

# Mengatur ukuran kanvas grafik
plt.figure(figsize=(14, 5))
# --- Grafik Akurasi (Accuracy) ---
plt.subplot(1, 2, 1)
# Catatan: Jika kamu memakai metrik dengan nama berbeda, 
# ubah 'accuracy' menjadi 'acc' sesuai versi Keras/TensorFlow.
plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# --- Grafik Error (Loss) ---
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

# Tampilkan grafik
plt.show()

In [ ]:
print("Melakukan prediksi...")
Y_pred_prob = model.predict(X)
Y_pred_classes = np.argmax(Y_pred_prob, axis=1)

if len(Y.shape) > 1 and Y.shape[1] > 1:
    Y_true_classes = np.argmax(Y, axis=1)
else:
    Y_true_classes = Y

print("\n--- Classification Report ---")
print(classification_report(Y_true_classes, Y_pred_classes))
cm = confusion_matrix(Y_true_classes, Y_pred_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Kelas Prediksi (Predicted)')
plt.ylabel('Kelas Aktual (True)')
plt.show()